# 在 `netsol/resume-score-details` 上单卡微调 Gemma 4 E4B-it

本 Notebook 仿照情绪分类微调文件的结构，改为使用 Hugging Face 上的 `netsol/resume-score-details` 数据集，对 Gemma 4 E4B-it 做 LoRA 微调，使模型生成“AI时代计算机相关岗位职业匹配模拟器”的结构化报告 JSON。

与原情绪分类 Notebook 不同：

1. **数据集改为 Hugging Face 的 `netsol/resume-score-details`**，因为当前目标数据集不在原 Notebook 使用的 ModelScope 镜像中。
2. **任务从情绪分类改为职业匹配报告生成**，输出不再是单个标签，而是固定字段的 JSON。
3. **评估指标改为生成任务指标**：JSON 合法率、字段完整率、匹配分合法率、有效样本数。
4. **保留单卡、BF16、LoRA、ModelScope 下载 Gemma 模型**，不使用 `bitsandbytes` 4bit，适配 AMD / ROCm 环境。
5. **不加入 RAG、岗位图谱、外部爬取数据**，只做本数据集相关的最小可运行微调。

> 注意：如果你使用 AMD ROCm，PyTorch 中仍然通过 `torch.cuda` 访问 GPU，这是 PyTorch 的统一接口。

## 1. 安装依赖

这里只安装本 Notebook 必需的包：

- `modelscope`：从魔搭下载 Gemma 模型。
- `datasets`：从 Hugging Face 加载 `netsol/resume-score-details`。
- `transformers`：加载 Gemma 模型和 tokenizer。
- `trl`：使用 `SFTTrainer` 做指令微调。
- `peft`：配置 LoRA。
- `pandas` / `tqdm`：保存和查看评估结果。

如果环境已经装好，可以跳过本单元。

In [ ]:
# AMD / ROCm 云环境通常已预装匹配的 torch。这里不使用 -U，避免把 ROCm torch 升级成 CUDA 版。
!uv pip install modelscope transformers accelerate datasets trl peft pandas tqdm sentencepiece protobuf --no-cache -i https://mirrors.cloud.tencent.com/pypi/simple/

## 2. 导入依赖与全局配置

默认使用单卡 BF16 LoRA 微调。显存不够时优先降低：

```python
TRAIN_LIMIT
EVAL_LIMIT
per_device_train_batch_size
max_length
```

如果显卡不支持 BF16，把：

```python
MODEL_DTYPE = torch.float16
BF16 = False
FP16 = True
```

In [ ]:
import os

import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import DatasetDict, Dataset
from modelscope import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")

# -----------------------------
# 基础配置
# -----------------------------
# 魔搭上的 Gemma 4 E4B-it 模型 ID。
MODELSCOPE_MODEL_ID = "google/gemma-4-E4B-it"

# 本地数据集路径（已下载好的 JSON 文件目录）
LOCAL_DATASET_DIR = "./datasets/resume-score-details"

# 微调输出目录。
OUTPUT_DIR = "./gemma4-it-resume-matching-lora-single-gpu"

# 数据量控制。先用小数据跑通，确认没问题后再加大或设为 None。
TRAIN_LIMIT = None
VALIDATION_LIMIT = 120
EVAL_LIMIT = 60

SEED = 42
MODEL_DTYPE = torch.bfloat16
BF16 = True
FP16 = False

# 分类标签
LABEL_MATCHED = "matched"
LABEL_MISMATCHED = "mismatched"

SYSTEM_PROMPT = """你是一个简历-岗位匹配分类器。
根据给定的简历和岗位描述，判断简历是否与岗位匹配。
只输出 "matched" 或 "mismatched"，不要输出其他内容。"""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./datasets", exist_ok=True)

print("torch version:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("HIP version:", getattr(torch.version, "hip", None))

if "+cu" in torch.__version__ and not torch.cuda.is_available():
    raise RuntimeError("当前环境安装的是 CUDA 版 torch，且 GPU 不可用。AMD/ROCm 云环境请恢复 ROCm 版 torch 后再继续。")
if torch.cuda.is_available() and getattr(torch.version, "hip", None) is None:
    print("Warning: 当前 torch 未显示 HIP 版本。如果这是 AMD 云环境，请确认没有误装 CUDA 版 torch。")

## 3. 固定随机种子

固定随机种子可以让数据 shuffle、LoRA 初始化和训练过程尽量可复现。

In [ ]:
def setup_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

setup_seed(SEED)

## 4. 从魔搭 ModelScope 下载 Gemma 模型

模型仍然沿用原 Notebook 的 ModelScope 下载方式，不需要从 Hugging Face 下载模型权重。

如果是第一次访问 Gemma 系列模型，可能需要先在 ModelScope 页面登录并接受许可协议。

In [ ]:
print("Downloading model from ModelScope...")
print("ModelScope model id:", MODELSCOPE_MODEL_ID)

model_dir = snapshot_download(
    MODELSCOPE_MODEL_ID,
    cache_dir="./models",
)

print("Downloaded model dir:", model_dir)
LOCAL_MODEL_DIR = model_dir

## 5. 加载本地 `netsol/resume-score-details` 数据集

直接从本地 JSON 文件加载数据集，不使用任何在线下载。

数据集路径：`./datasets/resume-score-details/`，包含原始 JSON 数据文件。

加载流程：
1. 读取目录下所有 JSON 文件
2. 解析为 Python 字典列表
3. 转换为 HuggingFace Dataset 对象
4. 按 90% / 10% 切分 train / validation

In [ ]:
import glob
from datasets import Dataset

print("Loading dataset from local JSON files...")
print("Local dataset dir:", LOCAL_DATASET_DIR)

# 读取目录下所有 JSON 文件
json_files = sorted(glob.glob(f"{LOCAL_DATASET_DIR}/*.json"))
print(f"Found {len(json_files)} JSON files")

if len(json_files) == 0:
    raise RuntimeError(f"No JSON files found in {LOCAL_DATASET_DIR}")

# 加载所有 JSON 数据
all_samples = []
for fp in tqdm(json_files, desc="Loading JSON files"):
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)
    # 每个文件可能是单个样本（dict）或多个样本（list）
    if isinstance(data, list):
        all_samples.extend(data)
    elif isinstance(data, dict):
        all_samples.append(data)

print(f"Total samples loaded: {len(all_samples)}")

# 转换为 HuggingFace Dataset
full_dataset = Dataset.from_list(all_samples)
print("Full dataset:", full_dataset)
print("Columns:", full_dataset.column_names)

# 按 90/10 切分 train / validation
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=SEED)
raw_dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

def maybe_limit(split, limit):
    split = split.shuffle(seed=SEED)
    if limit is None:
        return split
    return split.select(range(min(limit, len(split))))

dataset = DatasetDict({
    "train": maybe_limit(raw_dataset["train"], TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
})

print(dataset)
print("example keys:", dataset["train"][0].keys())

## 6. 构造简历-岗位匹配分类数据

将数据集转换为二分类任务：
- 输入：resume + job_description
- 输出：matched 或 mismatched

根据 output.scores.aggregated_scores 判断匹配度：
- macro_scores 和 micro_scores 的平均值 >= 5（假设10分制）认为 matched
- 否则为 mismatched

In [ ]:
def safe_get(obj, path, default=None):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

def get_match_label(sample):
    """根据aggregated_scores判断是否匹配"""
    aggregated = safe_get(sample, ["output", "scores", "aggregated_scores"], {})
    if not isinstance(aggregated, dict):
        return LABEL_MISMATCHED
    
    macro_score = aggregated.get("macro_scores")
    micro_score = aggregated.get("micro_scores")
    
    # 计算平均分（假设10分制）
    scores = []
    if macro_score is not None:
        try:
            scores.append(float(macro_score))
        except:
            pass
    if micro_score is not None:
        try:
            scores.append(float(micro_score))
        except:
            pass
    
    if not scores:
        return LABEL_MISMATCHED
    
    avg_score = sum(scores) / len(scores)
    # 5分及以上认为匹配
    return LABEL_MATCHED if avg_score >= 5.0 else LABEL_MISMATCHED

def is_valid_sample(sample):
    """过滤无效样本"""
    # 过滤掉 valid_resume_and_jd = false 的样本
    output = sample.get("output", {})
    if isinstance(output, dict) and output.get("valid_resume_and_jd") is False:
        return False
    
    # 确保有基本的输入数据
    input_data = sample.get("input", {})
    if not isinstance(input_data, dict):
        return False
    
    resume = input_data.get("resume", "")
    jd = input_data.get("job_description", "")
    
    return (isinstance(resume, str) and isinstance(jd, str) and 
            len(resume.strip()) >= 50 and len(jd.strip()) >= 50)

def to_classification_format(sample):
    """转换为分类格式"""
    input_data = sample.get("input", {})
    resume = input_data.get("resume", "")
    jd = input_data.get("job_description", "")
    
    label = get_match_label(sample)
    
    user_text = f"简历：\n{resume}\n\n岗位描述：\n{jd}"
    
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ],
        "completion": [
            {"role": "assistant", "content": label},
        ],
    }

# 过滤并转换数据
filtered_dataset = DatasetDict({
    split: dataset[split].filter(is_valid_sample)
    for split in dataset.keys()
})

sft_dataset = filtered_dataset.map(
    to_classification_format,
    remove_columns=filtered_dataset["train"].column_names,
)

print("Filtered dataset:", filtered_dataset)
print("SFT dataset:", sft_dataset)
print("\nExample prompt:", sft_dataset["train"][0]["prompt"])
print("Example completion:", sft_dataset["train"][0]["completion"])

# 统计标签分布
def count_labels(split_data):
    labels = [item["completion"][0]["content"] for item in split_data]
    from collections import Counter
    return Counter(labels)

print("\nLabel distribution in train:", count_labels(sft_dataset["train"]))
print("Label distribution in validation:", count_labels(sft_dataset["validation"]))

## 7. 加载 tokenizer 和基础模型

这里从魔搭下载后的本地路径加载模型。

单卡版本要点：

- 不使用 `device_map="auto"` 做多卡切分。
- 不使用 `bitsandbytes` 4bit。
- 开启 `gradient_checkpointing=True` 节省显存。
- 如果 tokenizer 缺少 chat template，则从同一 ModelScope 仓库读取 `chat_template.jinja` 作为兜底。

In [ ]:
print("Loading tokenizer from:", LOCAL_MODEL_DIR)

# 修复 tokenizer_config.json 中 extra_special_tokens 格式问题
# 某些 transformers 版本要求 dict 格式，但 Gemma 模型可能保存为 list
_tokenizer_cfg_path = os.path.join(LOCAL_MODEL_DIR, "tokenizer_config.json")
if os.path.exists(_tokenizer_cfg_path):
    with open(_tokenizer_cfg_path, "r", encoding="utf-8") as f:
        _tok_cfg = json.load(f)
    if isinstance(_tok_cfg.get("extra_special_tokens"), list):
        print("Fixing extra_special_tokens: list -> dict")
        _tok_cfg["extra_special_tokens"] = {}
        with open(_tokenizer_cfg_path, "w", encoding="utf-8") as f:
            json.dump(_tok_cfg, f, ensure_ascii=False, indent=2)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_DIR,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

def _load_official_gemma_chat_template() -> str:
    template_dir = snapshot_download(
        MODELSCOPE_MODEL_ID,
        cache_dir="./models",
        allow_file_pattern=["chat_template.jinja"],
    )
    path = os.path.join(template_dir, "chat_template.jinja")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

if not getattr(tokenizer, "chat_template", None):
    print("Loading official chat_template.jinja...")
    tokenizer.chat_template = _load_official_gemma_chat_template()
else:
    print("tokenizer.chat_template already set, leaving as-is.")

_probe = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
print("chat_template probe output:\n" + _probe)

print("Loading base model from:", LOCAL_MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

base_model.to(device)
base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Base model loaded.")
print("Base model device:", next(base_model.parameters()).device)

## 8. 推理辅助函数

微调前后都用同一套推理函数。模型输出分类标签：matched 或 mismatched。

In [ ]:
def predict_match(model, tokenizer, resume, job_description, max_new_tokens=10):
    """预测简历和岗位是否匹配"""
    user_text = f"简历：\n{resume}\n\n岗位描述：\n{job_description}"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    
    device = next(model.parameters()).device
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    pred_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip().lower()
    
    # 提取标签
    if "matched" in pred_text and "mismatched" not in pred_text:
        return LABEL_MATCHED
    elif "mismatched" in pred_text:
        return LABEL_MISMATCHED
    else:
        return pred_text  # 返回原始输出用于调试

# 测试基础模型
test_sample = filtered_dataset["validation"][0]
test_input = test_sample.get("input", {})
test_resume = test_input.get("resume", "")
test_jd = test_input.get("job_description", "")

print("Testing base model prediction...")
print("Predicted:", predict_match(base_model, tokenizer, test_resume, test_jd))

## 9. 评估函数

本任务是二分类，评估指标：
- `accuracy`：准确率
- `precision`、`recall`、`f1`：针对 matched 类别
- `evaluated_examples`：实际评估样本数

In [ ]:
def evaluate_model(model, tokenizer, split="validation", limit=EVAL_LIMIT):
    raw_source = filtered_dataset[split]
    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    predictions = []
    labels = []
    
    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        input_data = ex.get("input", {})
        resume = input_data.get("resume", "")
        jd = input_data.get("job_description", "")
        
        # 真实标签
        true_label = get_match_label(ex)
        labels.append(true_label)
        
        # 预测标签
        pred_label = predict_match(model, tokenizer, resume, jd)
        predictions.append(pred_label)

    # 计算指标
    correct = sum(1 for p, l in zip(predictions, labels) if p == l)
    total = len(labels)
    accuracy = correct / total if total > 0 else 0.0
    
    # 计算 precision, recall, F1 (针对 matched 类别)
    tp = sum(1 for p, l in zip(predictions, labels) if p == LABEL_MATCHED and l == LABEL_MATCHED)
    fp = sum(1 for p, l in zip(predictions, labels) if p == LABEL_MATCHED and l == LABEL_MISMATCHED)
    fn = sum(1 for p, l in zip(predictions, labels) if p == LABEL_MISMATCHED and l == LABEL_MATCHED)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "evaluated_examples": total,
    }
    
    # 构造预测结果 DataFrame
    pred_df = pd.DataFrame({
        "true_label": labels,
        "predicted_label": predictions,
        "correct": [p == l for p, l in zip(predictions, labels)],
    })
    
    return metrics, pred_df

## 10. 微调前评估

先评估基础模型生成职业匹配报告 JSON 的能力，作为 LoRA 微调后的对比基线。

如果只想快速训练，可以把 `RUN_PRE_EVAL = False`。

In [ ]:
RUN_PRE_EVAL = True

if RUN_PRE_EVAL:
    pre_metrics, pre_preds = evaluate_model(base_model, tokenizer, split="validation", limit=EVAL_LIMIT)
else:
    pre_metrics, pre_preds = {}, pd.DataFrame()

pre_metrics

## 11. 配置 LoRA

沿用原 Notebook 的简洁配置：

```python
target_modules="all-linear"
```

这样可以避免手动匹配不同 Gemma 版本内部层名。

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

## 12. 定义训练参数

与原 Notebook 的主要不同：

- `max_length` 从情绪分类的短文本长度提高到 `4096`，因为简历和 JD 更长。
- `per_device_train_batch_size` 降为 `1`，避免长上下文 OOM。
- 仍然使用 `adamw_torch`，避免 AMD ROCm 下 `bitsandbytes` 优化器兼容问题。

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=20,
    num_train_epochs=2,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,

    bf16=BF16,
    fp16=FP16,
    tf32=False,

    max_length=4096,
    packing=False,
    completion_only_loss=True,

    remove_unused_columns=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

## 13. 开始 LoRA 微调

训练前会检查 LoRA 参数是否真的被挂上。如果可训练参数为 0，说明 `target_modules` 没匹配成功，需要调整 LoRA 配置。

In [ ]:
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainable_params = 0
total_params = 0
trainable_param_names = []

for name, param in trainer.model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        trainable_param_names.append(name)

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters were attached. Check target_modules before training.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable ratio: {100 * trainable_params / total_params:.4f}%")
print("Example trainable parameters:")
print(trainable_param_names[:20])

train_result = trainer.train()

trainer.model.eval()
trainer.model.config.use_cache = True
train_result

## 14. 保存 LoRA adapter 和 tokenizer

保存的是 LoRA adapter，不是完整大模型权重。

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print("Saved adapter and tokenizer to:", OUTPUT_DIR)

## 15. 微调后评估

训练后直接复用内存里的模型评估，避免重新加载导致显存碎片或 OOM。

In [ ]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True

post_metrics, post_preds = evaluate_model(ft_model, tokenizer, split="validation", limit=EVAL_LIMIT)
post_metrics

## 16. 对比微调前后效果

这里合并微调前后的结构化生成指标，方便写报告。

In [ ]:
comparison_rows = []
if pre_metrics:
    comparison_rows.append({"stage": "pre_finetuning", **pre_metrics})
comparison_rows.append({"stage": "post_finetuning", **post_metrics})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 17. 手动测试微调后的模型

这里输入一个简历和岗位描述，观察模型是否能正确输出匹配标签。

In [ ]:
# 测试用例1: 匹配的简历和岗位
test_resume_matched = """本科软件工程，5年Python开发经验，熟悉Django、Flask框架，
有大型Web应用开发经验，熟悉MySQL、Redis、Elasticsearch。
曾负责电商平台后端架构设计和优化，日活百万级用户。"""

test_jd_matched = """招聘Python后端工程师，要求：
- 本科及以上学历，计算机相关专业
- 3年以上Python开发经验
- 熟悉Django或Flask框架
- 有Web应用开发经验
- 熟悉MySQL等关系数据库"""

print("测试用例1 (匹配):")
print("预测结果:", predict_match(ft_model, tokenizer, test_resume_matched, test_jd_matched))

print("\n" + "="*50 + "\n")

# 测试用例2: 不匹配的简历和岗位
test_resume_mismatched = """本科会计专业，3年财务工作经验，
熟悉财务报表编制、税务申报、成本核算。
持有中级会计师证书，熟练使用Excel、金蝶财务软件。"""

test_jd_mismatched = """招聘Python后端工程师，要求：
- 本科及以上学历，计算机相关专业
- 3年以上Python开发经验
- 熟悉Django或Flask框架"""

print("测试用例2 (不匹配):")
print("预测结果:", predict_match(ft_model, tokenizer, test_resume_mismatched, test_jd_mismatched))

## 18. 保存评估结果

输出文件会保存到 `OUTPUT_DIR` 中，方便后续写报告或对比。

In [ ]:
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "classification_before_after_metrics.csv"), index=False)
pre_preds.to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_predictions.csv"), index=False)
post_preds.to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_predictions.csv"), index=False)

print("Saved all outputs to:", OUTPUT_DIR)

## 19. 可选：重新加载本地 LoRA adapter 推理

以后不想重新训练，可以用下面代码重新加载：

1. 从魔搭下载的本地基础模型目录加载 base model。
2. 从 `OUTPUT_DIR` 加载 LoRA adapter。
3. 使用同一个 `generate_report()` 做推理。

In [ ]:
RUN_RELOAD_TEST = False

if RUN_RELOAD_TEST:
    reload_tokenizer = AutoTokenizer.from_pretrained(
        OUTPUT_DIR,
        use_fast=True,
        trust_remote_code=True,
    )
    if reload_tokenizer.pad_token is None:
        reload_tokenizer.pad_token = reload_tokenizer.eos_token

    reload_base_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_DIR,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    reload_base_model.to(device)

    reload_model = PeftModel.from_pretrained(
        reload_base_model,
        OUTPUT_DIR,
    )
    reload_model.eval()

    test_resume = "本科计算机专业，3年Java开发经验"
    test_jd = "招聘Java工程师，要求本科及以上，2年以上开发经验"
    print(predict_match(reload_model, reload_tokenizer, test_resume, test_jd))

## 20. 常见问题

### 1. 本notebook做什么任务？

**简历-岗位匹配二分类任务**：
- 输入：简历 + 岗位描述
- 输出：matched（匹配）或 mismatched（不匹配）
- 数据集：netsol/resume-score-details (1,031个样本)
- 方法：Gemma 4 E4B-it + LoRA微调

### 2. 如何判断匹配/不匹配？

根据数据集中的 `output.scores.aggregated_scores`：
- macro_scores 和 micro_scores 的平均值 >= 5.0 → matched
- 平均值 < 5.0 → mismatched
- 假设评分为10分制

### 3. 数据过滤规则

自动过滤：
- `valid_resume_and_jd = false` 的样本（142个无效样本）
- 简历或岗位描述长度 < 50字符的样本

### 4. 评估指标

- **accuracy**: 整体准确率
- **precision**: 预测为matched的准确率
- **recall**: 真实matched被正确识别的比例
- **f1**: precision和recall的调和平均

### 5. 显存不够怎么办？

优先调整：
```python
TRAIN_LIMIT = 300
VALIDATION_LIMIT = 50
EVAL_LIMIT = 20
per_device_train_batch_size = 1
gradient_accumulation_steps = 16
```

### 6. 标签不平衡怎么办？

数据集中matched和mismatched样本可能不平衡。运行Cell 6后会打印标签分布。
如果严重不平衡，可以考虑：
- 使用类别权重
- 下采样多数类或上采样少数类
- 调整评估指标（更关注F1而非accuracy）

当前实现是最小baseline，未处理类别不平衡问题。